# Final PCR Prediction (Classification)

In [23]:
import pandas as pd
import numpy as np

import joblib, csv

### Preprocessing

#### Before Preprocessing

In [24]:
dataset_x_all = pd.read_csv("./datasets/FinalTestDataset2025.csv")
dataset_x_all

,ID,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002219,47.0,1,1,0,0,3,2,1,1,...,0.498350,0.498350,3.144594,0.003447,8.257693e+06,150.048587,0.001753,0.037110,0.001369,0.001513
1,TRG002222,41.0,1,1,0,0,3,2,1,0,...,0.622381,0.622381,2.061654,0.006535,1.568442e+06,26.484938,0.009649,0.019352,0.000321,0.008285
2,TRG002223,53.0,0,0,0,1,2,1,1,1,...,0.412482,0.412482,3.440353,0.005391,2.656925e+06,174.606929,0.001594,0.075152,0.005255,0.001444
3,TRG002235,46.0,1,1,0,0,2,1,1,1,...,0.378333,0.378333,3.531715,0.007102,1.714787e+06,96.787378,0.002772,0.053377,0.002666,0.002406
4,TRG002240,39.0,0,0,1,0,2,2,1,1,...,0.524767,0.524767,2.186214,0.007896,5.104793e+05,12.789071,0.020072,0.023140,0.000463,0.017172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,TRG002953,62.3,0,0,1,0,2,1,1,0,...,0.550099,0.550097,2.632664,0.004373,1.583970e+06,91.084710,0.002972,0.042656,0.002168,0.002828
129,TRG002955,46.1,0,0,0,1,3,3,1,1,...,0.439568,0.439568,3.056046,0.001339,1.671271e+07,79.989003,0.003282,0.024716,0.000812,0.003078
130,TRG002956,44.3,0,0,0,1,3,2,1,0,...,0.450758,0.450758,2.820884,0.003774,4.493563e+06,144.506524,0.001912,0.053334,0.003453,0.001759
131,TRG002957,37.1,0,0,0,1,3,3,1,1,...,0.384860,0.384860,3.680654,0.003699,4.054944e+06,250.108074,0.001168,0.091763,0.008428,0.001107


#### Type-specific preprocessing
1. Drop ID
2. Change 'original_shape_VoxelVolume' to float
3. Get categorical/continuous column names
4. Replace 999 with NaN

In [25]:
ids = dataset_x_all["ID"]
dataset_x_all = dataset_x_all.drop("ID",axis=1)

dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)

categorical_names = dataset_x_all.select_dtypes(np.int64).columns
continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

#### General Preprocessing
1. IQR Outlier Removal
2. Cap categorical data range (based on definitional range)
3. KNN Imputation
4. Z-scaling / Normalisation
5. Convert Categorical float columns back to int

In [26]:
Xs = dataset_x_all.copy()

lower_quartile = Xs[continuous_names].quantile(0.25)
upper_quartile = Xs[continuous_names].quantile(0.75)
iqr = upper_quartile - lower_quartile

lower_bound = lower_quartile - 1.5 * iqr
upper_bound = upper_quartile + 1.5 * iqr

Xs[continuous_names] = np.where(Xs[continuous_names] < lower_bound, lower_bound, Xs[continuous_names])
Xs[continuous_names] = np.where(Xs[continuous_names] > upper_bound, upper_bound, Xs[continuous_names])

In [27]:
categorical_ranges = {
    "pCR (outcome)": (0,1),
    "ER": (0, 1),
    "PgR": (0, 1),
    "HER2": (0, 1),
    "TrippleNegative": (0, 1),
    "ChemoGrade": (1, 3),
    "Proliferation": (1, 3),
    "HistologyType": (1, 2),
    "LNStatus": (0, 1),
    "TumourStage": (1, 4),
    "Gene": (0, 1)
}

In [28]:
data_train_cat = pd.concat([Xs[categorical_names]], axis=1)
for col in data_train_cat.columns:
    
    # Truncate data to be within range for categorical features
    data_train_cat[col] = np.where(data_train_cat[col] > categorical_ranges[col][1], categorical_ranges[col][1],  data_train_cat[col])
    data_train_cat[col] = np.where(data_train_cat[col] < categorical_ranges[col][0], categorical_ranges[col][0],  data_train_cat[col])

In [29]:
imputer_knn = joblib.load('./exports/pcr_imputer_knn.pkl')
z_scaler = joblib.load('./exports/pcr_z_scaler_knn.pkl')

Xs = pd.DataFrame(imputer_knn.transform(Xs), columns=Xs.columns, index=Xs.index)
Xs[continuous_names] = z_scaler.transform(Xs[continuous_names])
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.437565,1.0,1.0,0.0,0.0,3.0,2.0,1.0,1.0,2.0,...,0.717659,0.760824,0.546091,0.076147,0.176502,0.298442,-0.778548,-0.505028,-0.724889,-0.770412
1,-1.032938,1.0,1.0,0.0,0.0,3.0,2.0,1.0,0.0,2.0,...,1.559607,1.629666,-0.988963,1.377665,-0.687816,-0.773089,0.097057,-0.928752,-0.908320,0.250798
2,0.157808,0.0,0.0,0.0,1.0,2.0,1.0,1.0,1.0,2.0,...,0.134768,0.159313,0.965327,0.895551,-0.547173,0.511410,-0.796125,0.402683,-0.044795,-0.780919
3,-0.536794,1.0,1.0,0.0,0.0,2.0,1.0,1.0,1.0,3.0,...,-0.097042,-0.079902,1.094832,1.616352,-0.668906,-0.163433,-0.665540,-0.116902,-0.497883,-0.635806
4,-1.231395,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,...,0.896985,0.945877,-0.812401,1.951142,-0.824515,-0.891858,1.252947,-0.838377,-0.883555,1.591010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,1.080635,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,3.0,...,1.068943,1.123316,-0.179562,0.466471,-0.685809,-0.212886,-0.643357,-0.372713,-0.585056,-0.572219
129,-0.526871,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,4.0,...,0.318634,0.349052,0.420576,-0.812696,1.268973,-0.309107,-0.608945,-0.800765,-0.822377,-0.534428
130,-0.705483,0.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0,2.0,...,0.394592,0.427436,0.087237,0.213800,-0.309861,0.250382,-0.760836,-0.117921,-0.360201,-0.733335
131,-1.419930,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,3.0,...,-0.052740,-0.034184,1.305952,0.182008,-0.366535,1.166147,-0.843361,0.799029,0.510702,-0.831749


In [30]:
# Convert categoricals back to int64 (as no more NaN/float64 which previously prevented this) 
Xs[categorical_names] = Xs[categorical_names].astype(np.int64)
Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.437565,1,1,0,0,3,2,1,1,2,...,0.717659,0.760824,0.546091,0.076147,0.176502,0.298442,-0.778548,-0.505028,-0.724889,-0.770412
1,-1.032938,1,1,0,0,3,2,1,0,2,...,1.559607,1.629666,-0.988963,1.377665,-0.687816,-0.773089,0.097057,-0.928752,-0.908320,0.250798
2,0.157808,0,0,0,1,2,1,1,1,2,...,0.134768,0.159313,0.965327,0.895551,-0.547173,0.511410,-0.796125,0.402683,-0.044795,-0.780919
3,-0.536794,1,1,0,0,2,1,1,1,3,...,-0.097042,-0.079902,1.094832,1.616352,-0.668906,-0.163433,-0.665540,-0.116902,-0.497883,-0.635806
4,-1.231395,0,0,1,0,2,2,1,1,2,...,0.896985,0.945877,-0.812401,1.951142,-0.824515,-0.891858,1.252947,-0.838377,-0.883555,1.591010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,1.080635,0,0,1,0,2,1,1,0,3,...,1.068943,1.123316,-0.179562,0.466471,-0.685809,-0.212886,-0.643357,-0.372713,-0.585056,-0.572219
129,-0.526871,0,0,0,1,3,3,1,1,4,...,0.318634,0.349052,0.420576,-0.812696,1.268973,-0.309107,-0.608945,-0.800765,-0.822377,-0.534428
130,-0.705483,0,0,0,1,3,2,1,0,2,...,0.394592,0.427436,0.087237,0.213800,-0.309861,0.250382,-0.760836,-0.117921,-0.360201,-0.733335
131,-1.419930,0,0,0,1,3,3,1,1,3,...,-0.052740,-0.034184,1.305952,0.182008,-0.366535,1.166147,-0.843361,0.799029,0.510702,-0.831749


### Feature Selection
Currently selected method: Recursive Feature Elimination | 10 Features

In [31]:
feature_sel = joblib.load('./exports/selected_features_pcr.pkl')
feature_sel

['ER',
 'PgR',
 'TrippleNegative',
 'LNStatus',
 'Gene',
 'original_firstorder_10Percentile',
 'original_firstorder_InterquartileRange',
 'original_firstorder_MeanAbsoluteDeviation',
 'original_glcm_MaximumProbability',
 'original_glrlm_RunEntropy',
 'HER2']

In [32]:
Xs = Xs[feature_sel]
Xs

,ER,PgR,TrippleNegative,LNStatus,Gene,original_firstorder_10Percentile,original_firstorder_InterquartileRange,original_firstorder_MeanAbsoluteDeviation,original_glcm_MaximumProbability,original_glrlm_RunEntropy,HER2
0,1,1,0,1,1,0.110250,-0.975251,-1.043144,0.644075,1.380899,0
1,1,1,0,0,1,0.596086,-0.886106,-1.019897,0.963692,0.745244,0
2,0,0,1,1,1,-0.348718,-0.083235,-0.013657,-0.156252,0.630389,0
3,1,1,0,1,0,0.186845,-0.041646,-0.035807,0.378333,-0.037157,0
4,0,0,0,1,0,0.634563,0.133193,0.162134,0.901064,-0.435646,1
...,...,...,...,...,...,...,...,...,...,...,...
128,0,0,0,0,1,0.354899,1.259442,1.599698,0.091849,0.188658,1
129,0,0,1,1,1,0.785963,0.049225,0.138107,0.556492,-0.059085,0
130,0,0,1,0,0,-0.302262,-1.343943,-1.375157,-0.062708,1.661260,0
131,0,0,1,1,1,-0.670325,-0.228107,-0.199254,-0.868279,0.853069,0


### Prediction
1. Load pickled models
2. Predict outcomes using chosen model
3. Concat ID column with outcomes
4. Write to csv

In [33]:
models: dict
models = joblib.load('./exports/pcr_models.pkl') # Load all models

# for key in models.keys():
#     print(f"{key}: {models[key].predict(Xs)}")


# LogReg seems to have greaatest acc in testing
    # use it as output

m = models['Logistic Regression']
m_results = m.predict(Xs)

# print(m_results)
# print(ids)


# write to csv using original ids (data order remains the same as no train/test split for example)
with open("./PCRPrediction.csv", "w", encoding="utf-8", newline='') as file:
    write = pd.concat([ids, pd.DataFrame(m_results, columns=['pCR (outcome)'])], ignore_index=False, sort=False, axis=1)
    write.to_csv(file, index=False)
    
write

,ID,pCR (outcome)
0,TRG002219,0
1,TRG002222,0
2,TRG002223,0
3,TRG002235,0
4,TRG002240,1
...,...,...
128,TRG002953,1
129,TRG002955,0
130,TRG002956,0
131,TRG002957,0
